# **Setup**

In [5]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [6]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [7]:
if IS_COLAB:
    !pip install optuna

import optuna

In [8]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on kaggle — storage at: /kaggle/working


# **Load Data**

In [9]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [10]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [11]:
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender

optimizer = ModelOptimizer("ItemKNN_asymmetric")

STUDY_NAME = ItemKNNCFRecommender.RECOMMENDER_NAME + '_asymmetric' + '_1'

In [12]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "asymmetric",
        "topK": optuna_trial.suggest_int("topK", 50, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "asymmetric_alpha": optuna_trial.suggest_float("asymmetric_alpha", 0.0, 1.0),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [13]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-19 19:39:27,308] A new study created in RDB with name: ItemKNNCFRecommender_asymmetric_1


  0%|          | 0/100 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1916.29 column/sec. Elapsed time 3.64 sec
  Fold 1/5 - Score: 0.17510917166270476
Similarity column 6969 (100.0%), 1964.48 column/sec. Elapsed time 3.55 sec
  Fold 2/5 - Score: 0.17523524459448134
Similarity column 6969 (100.0%), 1973.99 column/sec. Elapsed time 3.53 sec
  Fold 3/5 - Score: 0.17643768758433012
Similarity column 6969 (100.0%), 1896.71 column/sec. Elapsed time 3.67 sec
  Fold 4/5 - Score: 0.17510722572938378
Similarity column 6969 (100.0%), 1940.19 column/sec. Elapsed time 3.59 sec
  Fold 5/5 - Score: 0.175267580478044
[I 2025-11-19 19:42:12,524] Trial 0 finished with value: 0.1754313820097888 and parameters: {'topK': 405, 'shrink': 912, 'normalize': True, 'asymmetric_alpha': 0.2568270247507014, 'feature_weighting': 'none'}. Best is trial 0 with value: 0.1754313820097888.
Similarity column 6969 (100.0%), 1981.96 column/sec. Elapsed time 3.52 sec
  Fold 1/5 - Score: 0.19598152031359828
Similarity column 6969 (100.0%), 1965.78 column/sec. E

In [15]:
optuna.visualization.plot_optimization_history(optuna_study)

In [16]:
optuna.visualization.plot_param_importances(optuna_study)

In [17]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [20]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "asymmetric",
        "topK": optuna_trial.suggest_int("topK", 5, 80),
        "shrink": optuna_trial.suggest_int("shrink", 10, 50),
        "normalize": True,
        "asymmetric_alpha": optuna_trial.suggest_float("asymmetric_alpha", 0.08, 0.095),
        "feature_weighting": "TF-IDF",
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = ItemKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [21]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-19 22:33:11,475] A new study created in RDB with name: ItemKNNCFRecommender_asymmetric_1_refined


  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 6969 (100.0%), 1973.24 column/sec. Elapsed time 3.53 sec
  Fold 1/5 - Score: 0.24238345715652815
Similarity column 6969 (100.0%), 1996.84 column/sec. Elapsed time 3.49 sec
  Fold 2/5 - Score: 0.24174152300345533
Similarity column 6969 (100.0%), 1987.86 column/sec. Elapsed time 3.51 sec
  Fold 3/5 - Score: 0.24178506808733247
Similarity column 6969 (100.0%), 2014.65 column/sec. Elapsed time 3.46 sec
  Fold 4/5 - Score: 0.24103292017616923
Similarity column 6969 (100.0%), 2009.30 column/sec. Elapsed time 3.47 sec
  Fold 5/5 - Score: 0.24233793656237082
[I 2025-11-19 22:34:31,410] Trial 0 finished with value: 0.24185618099717118 and parameters: {'topK': 45, 'shrink': 27, 'asymmetric_alpha': 0.09435857749628664}. Best is trial 0 with value: 0.24185618099717118.
Similarity column 6969 (100.0%), 2019.44 column/sec. Elapsed time 3.45 sec
  Fold 1/5 - Score: 0.2430407511094984
Similarity column 6969 (100.0%), 1987.09 column/sec. Elapsed time 3.51 sec
  Fold 2/5 - Score: 0.243

In [22]:
optuna.visualization.plot_optimization_history(optuna_study)

In [23]:
optuna.visualization.plot_param_importances(optuna_study)

In [24]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- 1 tuning
Best Value: 0.2410396685281379
Best Params: {'topK': 50, 'shrink': 37, 'normalize': True, 'asymmetric_alpha': 0.0871226132417833, 'feature_weighting': 'TF-IDF'}
- 2 tuning
Best Value: 0.24515745634964875
Best Params: {'topK': 16, 'shrink': 45, 'asymmetric_alpha': 0.09085617207687702}

score: 0.24515745634964875

params: {'topK': 16, 'shrink': 45, 'normalize': True, 'asymmetric_alpha': 0.09085617207687702, 'feature_weighting': 'TF-IDF'}